In [2]:
## Running the Return Calculations
%run return_calc.ipynb

<class 'pandas.DataFrame'>
DatetimeIndex: 1343 entries, 2021-01-04 to 2026-05-08
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   BUFR    1343 non-null   float64
 1   IJR     1343 non-null   float64
 2   SMH     1343 non-null   float64
 3   SPY     1343 non-null   float64
 4   UGL     1343 non-null   float64
 5   VO      1343 non-null   float64
dtypes: float64(6)
memory usage: 73.4 KB


In [3]:
## Creating the Drawdown Calculation Function
def calculate_drawdown(value_series):
    running_peak = value_series.cummax()
    drawdown = (value_series / running_peak) - 1
    return drawdown

## Computing Portfolio and Benchmark Drawdown Calculations
portfolio_returns["Portfolio_Drawdown"] = calculate_drawdown(portfolio_returns["Portfolio_Value"])
portfolio_returns["Benchmark_Drawdown"] = calculate_drawdown(portfolio_returns["Benchmark_Value"])

In [4]:
## Rolling Risk Metrics

##  Creating the Rolling Volatility Computation Function
def calculate_rolling_volatility(return_series, window):
    return return_series.rolling(window).std() * np.sqrt(252)

## Computing the Rollling Volatility
portfolio_returns["Rolling_Vol_252d"] = calculate_rolling_volatility(
    portfolio_returns["Portfolio_Return"], window=252
)

## Creating the Rolling Return Computation Function
def calculate_rolling_return(return_series, window):
    return (1 + return_series).rolling(window).apply(np.prod, raw=True) - 1

## Computing the Rolling Return
portfolio_returns["Rolling_Return_252d"] = calculate_rolling_return(
    portfolio_returns["Portfolio_Return"], window=252
)

In [5]:
## Creating the Summary Performance Metric Calculation Function
def calculate_summary_metrics(portfolio_return_series, benchmark_return_series, portfolio_value_series, risk_free_rate):
    n_days = len(portfolio_return_series) ## number of trading days

    ## Total Return and Annualized Return and Volatility
    total_return = (portfolio_value_series.iloc[-1] / portfolio_value_series.iloc[0]) - 1
    annualized_return = (1 + total_return) ** (252 / n_days) - 1
    annualized_volatility = portfolio_return_series.std() * np.sqrt(252)

    ## Sharpe Ratio Computation
    sharpe_ratio = np.nan
    if annualized_volatility != 0:
        sharpe_ratio = (annualized_return - risk_free_rate) / annualized_volatility

    ## Max Drawdown Computation
    max_drawdown = calculate_drawdown(portfolio_value_series).min()

    ## Covariance and Beta Computation
    covariance = np.cov(portfolio_return_series, benchmark_return_series)[0, 1]
    benchmark_variance = np.var(benchmark_return_series)
    beta = covariance / benchmark_variance 
    
    if benchmark_variance == 0:
        np.nan
    
    benchmark_total_return = (1 + benchmark_return_series).prod() - 1
    benchmark_annualized_return = (1 + benchmark_total_return) ** (252 / len(benchmark_return_series)) - 1

    ## Alpha Computation
    alpha = annualized_return - (risk_free_rate + beta * (benchmark_annualized_return - risk_free_rate))

    ## Creating the dataframe structure
    metrics = pd.DataFrame({
        "Metrics": [
            "Total Return",
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Max Drawdown",
            "Beta",
            "Alpha"
        ],
        "Values": [
            total_return,
            annualized_return,
            annualized_volatility,
            sharpe_ratio,
            max_drawdown,
            beta,
            alpha
        ]
    })
    return metrics

## Computing the Summary Metrics
summary_metrics = calculate_summary_metrics(
    portfolio_returns["Portfolio_Return"].dropna(),
    portfolio_returns["Benchmark_Return"].dropna(),
    portfolio_returns["Portfolio_Value"].dropna(),
    risk_free_rate=RISK_FREE_RATE
)

## Outputting the Data
summary_metrics

,Metrics,Values
0,Total Return,1.572899
1,Annualized Return,0.194178
2,Annualized Volatility,0.173759
3,Sharpe Ratio,0.914354
4,Max Drawdown,-0.255219
5,Beta,0.898730
6,Alpha,0.051811


In [6]:
## Creating the Monthly Returns Table Computation Function
def calculate_monthly_returns(return_series):
    monthly_returns = (1 + return_series).resample("ME").prod() - 1
    return monthly_returns

## Computing the Monthly Portfolio and Benchmark Returns
monthly_portfolio = calculate_monthly_returns(portfolio_returns["Portfolio_Return"])
monthly_benchmark = calculate_monthly_returns(portfolio_returns["Benchmark_Return"])

## Putting this info into a dataframe
monthly_returns = pd.DataFrame({
    "Portfolio_Monthly_Return": monthly_portfolio,
    "Benchmark_Monthly_Return": monthly_benchmark
}).reset_index()

## Outputting the Data
monthly_returns.head()

,Date,Portfolio_Monthly_Return,Benchmark_Monthly_Return
0,2021-01-31,-0.003820,0.003471
1,2021-02-28,0.010984,0.027806
2,2021-03-31,0.018457,0.045399
3,2021-04-30,0.036910,0.052911
4,2021-05-31,0.041343,0.006566


In [7]:
## Creating the Asset Contribution Function
def calculate_asset_contribution(returns_wide_df, target_weights):
    weights = pd.Series(target_weights)
    weights = weights[returns_wide_df.columns]
    
    contribution_df = returns_wide_df.mul(weights, axis=1)
    cumulative_contribution = contribution_df.sum().sort_values(ascending=False)
    
    return contribution_df, cumulative_contribution

## Computing the Asset Contribution based on Portfolio Weight
daily_contribution, cumulative_contribution = calculate_asset_contribution(
    returns_wide, TARGET_WEIGHTS
)
cumulative_contribution

## Creating a Summary for Cumulative Contribution
contribution_summary = cumulative_contribution.reset_index()
contribution_summary.columns = ["ticker", "cumulative_contribution"]

contribution_summary

,ticker,cumulative_contribution
0,UGL,0.317482
1,SMH,0.300549
2,SPY,0.126150
3,VO,0.115291
4,IJR,0.093681
5,BUFR,0.083320


In [8]:
## Sanity Check for Daily Contribution
daily_contribution.head()

Ticker,BUFR,IJR,SMH,SPY,UGL,VO
Date,,,,,,
2021-01-05,0.001163,0.003196,0.002625,0.001033,0.001009,0.001904
2021-01-06,0.001504,0.007292,-0.000458,0.000897,-0.007054,0.003326
2021-01-07,0.000000,0.001585,0.006189,0.002229,-0.001040,0.002869
2021-01-08,-0.000069,-0.001233,-0.000539,0.000855,-0.014091,0.000905
2021-01-11,-0.000291,0.000691,0.002247,-0.001011,-0.000375,-0.000526
